In [ ]:
pip install numpy scikit-learn matplotlib scipy

In [1]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

In [2]:
CORPUS_FOLDER = r"C:\Users\刘嗝嗝\Desktop\ch_cgb"
TOP_N_BIGRAMS = 200

In [3]:
def clean_text(raw):
    text = re.sub(r"<[^>]+>", "", raw)           # remove HTML tags
    text = re.sub(r"^#{1,6}.*$", "", text, flags=re.MULTILINE)  # remove headings
    text = re.sub(r"^-{3,}$", "", text, flags=re.MULTILINE)     # remove dividers
    text = re.sub(r"[^\u4e00-\u9fff]", "", text)  # retain CJK characters only
    return text

In [ ]:
texts, labels = [], []

files = sorted(f for f in os.listdir(CORPUS_FOLDER) if f.endswith(".md"))
print(f"Found {len(files)} files")

for fname in files:
    num_match = re.search(r"(\d+)", fname)
    if not num_match:
        continue
    chapter_num = int(num_match.group(1))
    with open(os.path.join(CORPUS_FOLDER, fname), encoding="utf-8") as f:
        raw = f.read()
    cleaned = clean_text(raw)
    texts.append(cleaned)
    labels.append(chapter_num)
    print(f"Chapter {chapter_num:03d}: {len(cleaned)} characters")

print(f"\nTotal chapters loaded: {len(texts)}")

In [ ]:
# Count bigrams per chapter and across the full corpus
global_counts = Counter()
chapter_counts = []

for text in texts:
    counts = Counter()
    for i in range(len(text) - 1):
        bigram = text[i:i+2]
        counts[bigram] += 1
    chapter_counts.append(counts)
    global_counts.update(counts.keys())

# Select top 200 most frequent bigrams as features
top_bigrams = [bg for bg, _ in global_counts.most_common(TOP_N_BIGRAMS)]

print("Top 10 most frequent bigrams across all chapters:")
for bg, count in global_counts.most_common(10):
    print(f"  '{bg}': {count}")

# Build relative frequency matrix (120 chapters x 200 features)
matrix = np.zeros((len(texts), TOP_N_BIGRAMS))
for i, counts in enumerate(chapter_counts):
    total = sum(counts.values()) or 1
    for j, bg in enumerate(top_bigrams):
        matrix[i, j] = counts.get(bg, 0) / total

print(f"\nFeature matrix shape: {matrix.shape} ({len(texts)} chapters x {TOP_N_BIGRAMS} features)")

In [6]:
scaler = StandardScaler()
scaled = scaler.fit_transform(matrix)

In [ ]:
# Reduce to 2 dimensions
pca = PCA(n_components=2)
coords = pca.fit_transform(scaled)
var1, var2 = pca.explained_variance_ratio_ * 100

# Plot
fig, ax = plt.subplots(figsize=(12, 8))

for i, (ch, (x, y)) in enumerate(zip(labels, coords)):
    color = "#2166ac" if ch <= 80 else "#d6604d"
    ax.scatter(x, y, color=color, s=60, zorder=3)
    ax.annotate(str(ch), (x, y), fontsize=6.5,
                ha="center", va="bottom", color=color)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#2166ac",
           markersize=9, label="Chapters 1-80 (Cao Xueqin)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#d6604d",
           markersize=9, label="Chapters 81-120 (disputed)"),
]
ax.legend(handles=legend_elements, fontsize=10)
ax.set_xlabel(f"PC1 ({var1:.1f}% variance explained)", fontsize=11)
ax.set_ylabel(f"PC2 ({var2:.1f}% variance explained)", fontsize=11)
ax.set_title("PCA of Character Bigram Frequencies\nDream of the Red Chamber",
             fontsize=13, fontweight="bold")
ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
ax.axvline(0, color="grey", linewidth=0.5, linestyle="--")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(r"C:\Users\刘嗝嗝\Desktop\pca_plot.png", dpi=150)
plt.show()
print(f"PC1: {var1:.1f}% variance explained")
print(f"PC2: {var2:.1f}% variance explained")

In [ ]:
# Re-run PCA with three groups
fig, ax = plt.subplots(figsize=(12, 8))

for i, (ch, (x, y)) in enumerate(zip(labels, coords)):
    if ch <= 40:
        color = "#2166ac"    # dark blue: chapters 1-40
        marker = "o"
    elif ch <= 80:
        color = "#92c5de"    # light blue: chapters 41-80
        marker = "o"
    else:
        color = "#d6604d"    # red: chapters 81-120
        marker = "o"
    ax.scatter(x, y, color=color, s=60, zorder=3)
    ax.annotate(str(ch), (x, y), fontsize=6.5,
                ha="center", va="bottom", color=color)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#2166ac",
           markersize=9, label="Chapters 1-40"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#92c5de",
           markersize=9, label="Chapters 41-80"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#d6604d",
           markersize=9, label="Chapters 81-120 (disputed)"),
]
ax.legend(handles=legend_elements, fontsize=10)
ax.set_xlabel(f"PC1 ({var1:.1f}% variance explained)", fontsize=11)
ax.set_ylabel(f"PC2 ({var2:.1f}% variance explained)", fontsize=11)
ax.set_title("PCA of Character Bigram Frequencies — Three-group Comparison\nDream of the Red Chamber",
             fontsize=13, fontweight="bold")
ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
ax.axvline(0, color="grey", linewidth=0.5, linestyle="--")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(r"C:\Users\刘嗝嗝\Desktop\pca_three_groups.png", dpi=150)
plt.show()

In [ ]:
# Compute pairwise Euclidean distances and run Ward clustering
condensed = pdist(scaled, metric="euclidean")
linked = linkage(condensed, method="ward")

# Plot dendrogram
fig, ax = plt.subplots(figsize=(20, 7))
dendrogram(
    linked,
    labels=[str(ch) for ch in labels],
    ax=ax,
    leaf_rotation=90,
    leaf_font_size=7,
    above_threshold_color="grey"
)

# Colour leaf labels: blue for chapters 1-80, red for 81-120
for lbl in ax.get_xmajorticklabels():
    ch = int(lbl.get_text())
    lbl.set_color("#2166ac" if ch <= 80 else "#d6604d")

ax.set_title(
    "Hierarchical Clustering — Dream of the Red Chamber\n"
    "Blue: Chapters 1-80 | Red: Chapters 81-120",
    fontsize=12, fontweight="bold"
)
ax.set_ylabel("Euclidean Distance", fontsize=10)
ax.set_xlabel("Chapter Number", fontsize=10)

plt.tight_layout()
plt.savefig(r"C:\Users\刘嗝嗝\Desktop\dendrogram.png", dpi=150, bbox_inches="tight")
plt.show()
print("Done. Both figures saved to Desktop.")

In [ ]:
# Extract the two groups
indices_first80 = [i for i, ch in enumerate(labels) if ch <= 80]
indices_last40  = [i for i, ch in enumerate(labels) if ch >= 81]

scaled_first80 = scaled[indices_first80]
scaled_last40  = scaled[indices_last40]

labels_first80 = [labels[i] for i in indices_first80]
labels_last40  = [labels[i] for i in indices_last40]

# Compute cross-group distance matrix (80 x 40)
dist_cross = euclidean_distances(scaled_first80, scaled_last40)

# Plot
fig, ax = plt.subplots(figsize=(16, 20))
im = ax.imshow(dist_cross, cmap="RdYlBu_r", aspect="auto")
plt.colorbar(im, ax=ax, label="Euclidean Distance")

ax.set_xticks(range(len(labels_last40)))
ax.set_yticks(range(len(labels_first80)))
ax.set_xticklabels(labels_last40, fontsize=7, rotation=90)
ax.set_yticklabels(labels_first80, fontsize=7)

ax.set_xlabel("Chapters 81–120 (disputed)", fontsize=11)
ax.set_ylabel("Chapters 1–80 (Cao Xueqin)", fontsize=11)
ax.set_title(
    "Cross-group Stylistic Distance\nChapters 1–80 vs Chapters 81–120",
    fontsize=13, fontweight="bold"
)

plt.tight_layout()
plt.savefig(r"C:\Users\刘嗝嗝\Desktop\heatmap_cross.png", dpi=150)
plt.show()
print("Cross-group heatmap saved.")